<a href="https://colab.research.google.com/github/lestojas/segmentation/blob/claude/lestojas-segmentation-direction-o9c5p8/colab/train_crack_direction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crack Detection, Segmentation & Direction Classification

This notebook trains and evaluates a three-model pipeline on the `lestojas/segmentation`
crack dataset, using **YOLO11** (Ultralytics' current generation — the most advanced
real-time detection/segmentation/classification architecture that can be trained
end-to-end in a single Colab session):

1. **YOLO11-seg** (`yolo11m-seg`) — instance segmentation model. Answers *"is there a
   crack, and what shape is it?"* (crack vs. no-crack detection + pixel-accurate mask
   segmentation).
2. **YOLO11-cls** (`yolo11m-cls`) — image classification model. Answers *"which
   direction does the crack run?"* (Horizontal / Vertical / Diagonal / Mixed), trained
   on the direction labels already computed in the repo (`*/_direction_labels.csv`,
   derived via PCA on the ground-truth segmentation polygons — see
   `DIRECTION_LABELS.md`).
3. A smaller **baseline pair** (`yolo11n-seg` / `yolo11n-cls`) trained on the identical
   data and test set, purely so the results can be benchmarked against something real
   with a proper paired statistical test — not just against guessing.
4. A larger **third pair** (`yolo11l-seg` / `yolo11l-cls`) — same idea, but scaling
   model capacity *up* instead of down, aimed at improving segmentation and direction
   accuracy specifically (a `-seg` checkpoint always does detection+segmentation
   jointly, so this is the only way to try to improve segmentation without retraining
   detection separately).

Every training cell below **skips straight to evaluation if a matching
`runs/<run_name>/weights/best.pt` already exists** (e.g. from an earlier session whose
weights you re-uploaded — see the guidance in Section 0), so re-running this notebook
doesn't force a redundant multi-hour retrain. Set `FORCE_RETRAIN = True` in the config
cell to always retrain from scratch instead.

A final evaluation section reports all three requested accuracies:
- **Crack vs. no-crack detection accuracy**
- **Crack shape segmentation accuracy** (mask IoU / mAP)
- **Crack direction classification accuracy**

Section 10 then packages the key results into **eight research-paper-ready tables**
(Table 0: image counts per split, for a Methods section; Table 1: the test split's
composition in detail, for a Results section; Tables 2-6: descriptive performance --
detection confusion matrix, detection image-level metrics, detection box-level
metrics, segmentation, direction, with one column per model actually trained; Table 7:
symmetric pairwise significance tests between every pair of trained models, with no
model singled out as 'the' baseline) — exported as CSV and LaTeX and zipped for
download.

> **ℹ️ About the negative images:** every split includes genuine crack-free negatives
> (see `DIRECTION_LABELS.md`) — real crops from a crack-containing photo's crack-free
> region, not synthetic images. `train`/`valid` have ~15% negatives; the held-out
> **test** split is specifically rebalanced to near-parity (89 crack / 89 crack-free)
> so the detection confusion matrix (TP/FP/FN/TN) and specificity are meaningful
> rather than computed from a handful of negatives.

**Before running:** `Runtime → Change runtime type → GPU` (T4 or better).

## 0. Setup

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — go to Runtime > Change runtime type > GPU before continuing.')

In [ ]:
!pip install -q ultralytics scikit-learn seaborn

In [ ]:
# --- Configuration ---------------------------------------------------------
REPO_URL = 'https://github.com/lestojas/segmentation.git'
BRANCH   = 'claude/lestojas-segmentation-direction-o9c5p8'  # switch to 'main' once this branch is merged

SEG_MODEL   = 'yolo11m-seg.pt'   # s/m/l/x — bigger = more accurate, slower. m is a good default for Colab.
CLS_MODEL   = 'yolo11m-cls.pt'
IMG_SIZE    = 640
SEG_EPOCHS  = 100
CLS_EPOCHS  = 50
BATCH       = -1   # -1 = Ultralytics auto-picks the largest batch that fits in GPU memory
CONF_THRES  = 0.25 # confidence threshold used for the 'is a crack detected in this image' metric

TRAIN_BASELINE_COMPARISON = True  # trains a second, smaller model for benchmarking in Tables 2-6 and Table 7.
                                   # Set False to skip it and save time -- Tables 0-1 don't need it.
BASELINE_SEG_MODEL = 'yolo11n-seg.pt'  # smallest YOLO11 seg variant -- same architecture family as SEG_MODEL,
BASELINE_CLS_MODEL = 'yolo11n-cls.pt'  # so the comparison isolates model capacity, not architecture family

TRAIN_THIRD_MODEL = True  # trains a third, LARGER model -- same idea as the baseline, but scaling capacity UP
                           # instead of down, aimed at improving the segmentation/direction tasks specifically.
THIRD_SEG_MODEL = 'yolo11l-seg.pt'
THIRD_CLS_MODEL = 'yolo11l-cls.pt'

FORCE_RETRAIN = False  # if runs/<run_name>/weights/best.pt already exists for a given model (e.g. you
                        # uploaded a previously-trained checkpoint, or re-ran this notebook in the same
                        # session), training for that model is skipped and the existing weights are reused
                        # as-is. Set True to always retrain every model from scratch regardless.

SAVE_TO_DRIVE = False  # set True to copy trained weights to your Google Drive at the end

### Recovering previously-trained weights (optional -- skips retraining)

Every training cell in Sections 3-4 checks whether `runs/<run_name>/weights/best.pt`
already exists before training, and skips straight past it if so (set
`FORCE_RETRAIN = True` above to always retrain regardless). If you already trained
some of these models in an earlier session and still have the `.pt` files saved
somewhere, you can skip retraining just those models:

1. Run the repo-clone cell below first, so the working directory exists to upload into.
2. For each model you want to skip, create its `weights/` folder and drop `best.pt`
   into it, e.g. for the main segmentation model:
   ```python
   import os; os.makedirs('runs/crack_seg/weights', exist_ok=True)
   from google.colab import files
   uploaded = files.upload()  # pick your saved best.pt in the dialog
   os.rename(list(uploaded.keys())[0], 'runs/crack_seg/weights/best.pt')
   ```
   The run names to match are `crack_seg`, `crack_seg_baseline`, `crack_direction_cls`,
   and `crack_direction_cls_baseline`. (The third, larger model --
   `crack_seg_third` / `crack_direction_cls_third` -- is new in this notebook
   revision, so there's nothing to recover for it; it trains fresh the first time.)
3. If you used `SAVE_TO_DRIVE = True` in an earlier run instead, mount Drive
   (`from google.colab import drive; drive.mount('/content/drive')`) and copy from
   `MyDrive/crack_models/` into the matching `runs/<run_name>/weights/best.pt` path.

Either way, Sections 3-4 work correctly either path: found weights are reused as-is,
anything missing is trained normally.

In [ ]:
import os, shutil
DATA_DIR = '/content/segmentation'
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
!git clone --branch {BRANCH} --single-branch {REPO_URL} {DATA_DIR}
os.chdir(DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
import json, csv, math, random, zipfile, itertools
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from sklearn.metrics import confusion_matrix, classification_report
from scipy import stats
from IPython.display import display

from ultralytics import YOLO

SPLITS = ['train', 'valid', 'test']
random.seed(0)

## 1. Convert COCO segmentation annotations → YOLO-seg format

Ultralytics expects one `.txt` label file per image, one line per instance:
`class_id x1 y1 x2 y2 ... xn yn` with all coordinates normalized to `[0, 1]`. This
dataset has a single object class (`Cracks`), so every instance maps to class `0`.

In [ ]:
YOLO_SEG_DIR = Path('/content/yolo_seg_dataset')
if YOLO_SEG_DIR.exists():
    shutil.rmtree(YOLO_SEG_DIR)

def convert_split_to_yolo_seg(split):
    coco = json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json'))
    anns_by_image = defaultdict(list)
    for a in coco['annotations']:
        anns_by_image[a['image_id']].append(a)

    img_out = YOLO_SEG_DIR / split / 'images'
    lbl_out = YOLO_SEG_DIR / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for im in coco['images']:
        w, h = im['width'], im['height']
        src = Path(DATA_DIR) / split / im['file_name']
        dst = img_out / im['file_name']
        if not dst.exists():
            os.symlink(src, dst)

        lines = []
        for ann in anns_by_image.get(im['id'], []):
            for seg in ann.get('segmentation', []):
                if len(seg) < 6:
                    continue
                coords = []
                for i in range(0, len(seg), 2):
                    x = min(max(seg[i] / w, 0.0), 1.0)
                    y = min(max(seg[i + 1] / h, 0.0), 1.0)
                    coords.append(f'{x:.6f} {y:.6f}')
                lines.append('0 ' + ' '.join(coords))

        (lbl_out / (Path(im['file_name']).stem + '.txt')).write_text('\n'.join(lines))

    return len(coco['images']), sum(len(v) for v in anns_by_image.values())

for split in SPLITS:
    n_img, n_ann = convert_split_to_yolo_seg(split)
    print(f'{split}: {n_img} images, {n_ann} annotations converted')

In [ ]:
data_yaml = f'''
path: {YOLO_SEG_DIR}
train: train/images
val: valid/images
test: test/images
names:
  0: Cracks
'''
yaml_path = YOLO_SEG_DIR / 'data.yaml'
yaml_path.write_text(data_yaml)
print(data_yaml)

## 2. Build the direction-classification dataset

Ultralytics' classification trainer expects `train/<class_name>/*.jpg`,
`val/<class_name>/*.jpg` folders. We build that structure from the
`_direction_labels.csv` files already computed for this dataset (see
`DIRECTION_LABELS.md` for how those labels were derived).

In [ ]:
DIR_DATASET = Path('/content/direction_dataset')
if DIR_DATASET.exists():
    shutil.rmtree(DIR_DATASET)

SPLIT_TO_YOLO_CLS = {'train': 'train', 'valid': 'val', 'test': 'test'}
direction_rows = []

for split in SPLITS:
    with open(f'{DATA_DIR}/{split}/_direction_labels.csv') as f:
        rows = list(csv.DictReader(f))
    out_split = SPLIT_TO_YOLO_CLS[split]
    for r in rows:
        cls_dir = DIR_DATASET / out_split / r['direction']
        cls_dir.mkdir(parents=True, exist_ok=True)
        src = Path(DATA_DIR) / split / r['file_name']
        dst = cls_dir / r['file_name']
        if not dst.exists():
            os.symlink(src, dst)
        r['split'] = split
        direction_rows.append(r)

direction_df = pd.DataFrame(direction_rows)
direction_df['num_annotations'] = direction_df['num_annotations'].astype(int)
direction_df['angle_deg'] = pd.to_numeric(direction_df['angle_deg'], errors='coerce')
direction_df['elongation_ratio'] = pd.to_numeric(direction_df['elongation_ratio'], errors='coerce')
print(direction_df.groupby(['split', 'direction']).size().unstack(fill_value=0))

## 3. Train YOLO11-seg — crack detection + shape segmentation

In [ ]:
seg_model_path = Path('runs/crack_seg/weights/best.pt')
if FORCE_RETRAIN or not seg_model_path.exists():
    seg_model = YOLO(SEG_MODEL)
    seg_train_results = seg_model.train(
        data=str(yaml_path),
        epochs=SEG_EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        patience=20,      # stop early once val performance plateaus -- saves time, doesn't cost accuracy
        cache='ram',       # decode+cache images once instead of re-reading from disk every epoch (pure speed win)
        cos_lr=True,       # cosine LR schedule -- standard fine-tuning setting, usually a small free accuracy gain
        amp=True,          # mixed precision (default, made explicit) -- ~2x faster on a T4/A100 at equal accuracy
        project='runs',
        name='crack_seg',
        seed=0,
    )
else:
    print(f'{seg_model_path} already exists -- skipping training (FORCE_RETRAIN=False). '
          f'Delete it or set FORCE_RETRAIN=True above to retrain from scratch.')

### Baseline comparison model (optional)

Trains `yolo11n-seg` — the smallest YOLO11 segmentation variant — on the *exact same*
data, image size, and epoch budget as the main model above. Same architecture family,
only model capacity differs, so any gap between them isolates "does the bigger model
actually help" rather than confounding it with different hyperparameters or training
time. This is what feeds one of the columns in Tables 2-6 and lets Table 7
(Section 10) run genuine paired benchmark tests instead of just comparing to a
trivial guess.

In [ ]:
if TRAIN_BASELINE_COMPARISON:
    baseline_seg_model_path = Path('runs/crack_seg_baseline/weights/best.pt')
    if FORCE_RETRAIN or not baseline_seg_model_path.exists():
        baseline_seg_model = YOLO(BASELINE_SEG_MODEL)
        baseline_seg_model.train(
            data=str(yaml_path),
            epochs=SEG_EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            patience=20,
            cache='ram',
            cos_lr=True,
            amp=True,
            project='runs',
            name='crack_seg_baseline',
            seed=0,
        )
    else:
        print(f'{baseline_seg_model_path} already exists -- skipping training (FORCE_RETRAIN=False).')
else:
    print('TRAIN_BASELINE_COMPARISON is False -- skipping the baseline segmentation model.')

### Third, larger model (optional)

Trains `yolo11l-seg` — a larger YOLO11 segmentation variant than the main model — on
the *exact same* data, image size, and epoch budget. The main and baseline models above
already give strong crack-detection results; this one exists to test whether more
capacity specifically improves the weaker segmentation-shape and (via Section 4/7)
direction numbers, without touching the detection pipeline.

In [ ]:
if TRAIN_THIRD_MODEL:
    third_seg_model_path = Path('runs/crack_seg_third/weights/best.pt')
    if FORCE_RETRAIN or not third_seg_model_path.exists():
        third_seg_model = YOLO(THIRD_SEG_MODEL)
        third_seg_model.train(
            data=str(yaml_path),
            epochs=SEG_EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            patience=20,
            cache='ram',
            cos_lr=True,
            amp=True,
            project='runs',
            name='crack_seg_third',
            seed=0,
        )
    else:
        print(f'{third_seg_model_path} already exists -- skipping training (FORCE_RETRAIN=False).')
else:
    print('TRAIN_THIRD_MODEL is False -- skipping the third segmentation model.')

## 4. Train YOLO11-cls — crack direction classification

In [ ]:
cls_model_path = Path('runs/crack_direction_cls/weights/best.pt')
if FORCE_RETRAIN or not cls_model_path.exists():
    cls_model = YOLO(CLS_MODEL)
    cls_train_results = cls_model.train(
        data=str(DIR_DATASET),
        epochs=CLS_EPOCHS,
        imgsz=224,
        batch=BATCH if BATCH != -1 else 64,
        patience=20,
        cache='ram',
        cos_lr=True,
        amp=True,
        project='runs',
        name='crack_direction_cls',
        seed=0,
    )
else:
    print(f'{cls_model_path} already exists -- skipping training (FORCE_RETRAIN=False). '
          f'Delete it or set FORCE_RETRAIN=True above to retrain from scratch.')

### Baseline comparison model (optional)

Same idea as the segmentation baseline: `yolo11n-cls` (smallest variant) trained with
the same data and epoch budget as the main classifier, for Table 6's descriptives and Table 7's paired tests.

In [ ]:
if TRAIN_BASELINE_COMPARISON:
    baseline_cls_model_path = Path('runs/crack_direction_cls_baseline/weights/best.pt')
    if FORCE_RETRAIN or not baseline_cls_model_path.exists():
        baseline_cls_model = YOLO(BASELINE_CLS_MODEL)
        baseline_cls_model.train(
            data=str(DIR_DATASET),
            epochs=CLS_EPOCHS,
            imgsz=224,
            batch=BATCH if BATCH != -1 else 64,
            patience=20,
            cache='ram',
            cos_lr=True,
            amp=True,
            project='runs',
            name='crack_direction_cls_baseline',
            seed=0,
        )
    else:
        print(f'{baseline_cls_model_path} already exists -- skipping training (FORCE_RETRAIN=False).')
else:
    print('TRAIN_BASELINE_COMPARISON is False -- skipping the baseline direction classifier.')

### Third, larger model (optional)

`yolo11l-cls` (a larger variant than the main classifier), same data and epoch budget --
the direction-classification counterpart of the third segmentation model above.

In [ ]:
if TRAIN_THIRD_MODEL:
    third_cls_model_path = Path('runs/crack_direction_cls_third/weights/best.pt')
    if FORCE_RETRAIN or not third_cls_model_path.exists():
        third_cls_model = YOLO(THIRD_CLS_MODEL)
        third_cls_model.train(
            data=str(DIR_DATASET),
            epochs=CLS_EPOCHS,
            imgsz=224,
            batch=BATCH if BATCH != -1 else 64,
            patience=20,
            cache='ram',
            cos_lr=True,
            amp=True,
            project='runs',
            name='crack_direction_cls_third',
            seed=0,
        )
    else:
        print(f'{third_cls_model_path} already exists -- skipping training (FORCE_RETRAIN=False).')
else:
    print('TRAIN_THIRD_MODEL is False -- skipping the third direction classifier.')

## 5. Evaluate — crack vs. no-crack detection accuracy

`seg_model.val()` gives standard object-detection metrics on the held-out **test**
split (box precision/recall/mAP -- these are *box-level*, i.e. how well individual
predicted boxes line up with ground-truth boxes by IoU). We then build the
*image-level* confusion matrix that actually answers "crack vs. no-crack": for each
test image (a near-parity mix of crack and crack-free images -- see the note above), did
the model fire at least one crack detection above `CONF_THRES`? That gives TP/FP/FN/TN
and the standard classification metrics computed from them --
Accuracy = (TP+TN)/(TP+TN+FP+FN), Precision = TP/(TP+FP), Recall = TP/(TP+FN),
Specificity = TN/(TN+FP), F1 = 2·Precision·Recall/(Precision+Recall).

In [ ]:
best_seg = YOLO('runs/crack_seg/weights/best.pt')
seg_val = best_seg.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)

box_map50    = seg_val.box.map50
box_map50_95 = seg_val.box.map
box_precision = seg_val.box.mp
box_recall    = seg_val.box.mr
print(f'Box detection  — mAP50: {box_map50:.4f}  mAP50-95: {box_map50_95:.4f}  '
      f'Precision: {box_precision:.4f}  Recall: {box_recall:.4f}')

In [ ]:
# Image-level 'is there a detected crack in this image?' check
test_images_dir = YOLO_SEG_DIR / 'test' / 'images'
test_files = sorted(test_images_dir.glob('*.jpg'))

coco_test = json.load(open(f'{DATA_DIR}/test/_annotations.coco.json'))
gt_has_crack = defaultdict(bool)
for a in coco_test['annotations']:
    gt_has_crack[a['image_id']] = True
filename_to_gt = {im['file_name']: gt_has_crack.get(im['id'], False) for im in coco_test['images']}

y_true, y_pred = [], []
# Run inference once and keep it -- Section 6 reuses these same Results objects for
# mask-IoU matching instead of predicting over the test set a second time.
main_test_preds = best_seg.predict(source=[str(p) for p in test_files], conf=CONF_THRES,
                                    imgsz=IMG_SIZE, verbose=False)
for p in main_test_preds:
    fname = Path(p.path).name
    y_true.append(filename_to_gt.get(fname, False))
    y_pred.append(len(p.boxes) > 0)

y_true = np.array(y_true); y_pred = np.array(y_pred)

# Explicit confusion matrix: TP = crack present & detected, TN = crack-free & correctly
# said so, FP = crack-free but a crack was (wrongly) detected, FN = crack present but missed.
tp = int(np.sum(y_true & y_pred))
fn = int(np.sum(y_true & ~y_pred))
fp = int(np.sum(~y_true & y_pred))
tn = int(np.sum(~y_true & ~y_pred))

detection_accuracy = (tp + tn) / (tp + tn + fp + fn)
detection_precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
detection_recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
detection_specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
detection_f1 = (2 * detection_precision * detection_recall / (detection_precision + detection_recall)
                if (detection_precision + detection_recall) > 0 else np.nan)
n_pos = y_true.sum(); n_neg = (~y_true).sum()

print(f'Test images: {len(y_true)}  (crack-labeled: {n_pos}, crack-free: {n_neg})')
print(f'Confusion matrix -- TP: {tp}  FN: {fn}  FP: {fp}  TN: {tn}')
print(f'Accuracy: {detection_accuracy:.4f}  Precision: {detection_precision:.4f}  '
      f'Recall: {detection_recall:.4f}  Specificity: {detection_specificity:.4f}  '
      f'F1: {detection_f1:.4f}')

det_cm = np.array([[tp, fn], [fp, tn]])
plt.figure(figsize=(4.5, 4))
sns.heatmap(det_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted crack', 'Predicted no-crack'],
            yticklabels=['Actual crack', 'Actual no-crack'])
plt.title('Crack vs. no-crack -- confusion matrix'); plt.tight_layout(); plt.show()

### Baseline model comparison

Same test images, same `CONF_THRES`, the smaller `yolo11n-seg` model from Section 3 --
needed so Tables 2-4 can show its numbers alongside the other models', and
for Table 7's paired significance tests. Predictions are kept for reuse in Section 6
too, same as the main model above.

In [ ]:
if TRAIN_BASELINE_COMPARISON:
    best_seg_baseline = YOLO('runs/crack_seg_baseline/weights/best.pt')
    seg_val_baseline = best_seg_baseline.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)
    baseline_box_map50    = seg_val_baseline.box.map50
    baseline_box_map50_95 = seg_val_baseline.box.map
    baseline_box_precision = seg_val_baseline.box.mp
    baseline_box_recall    = seg_val_baseline.box.mr

    baseline_test_preds = best_seg_baseline.predict(source=[str(p) for p in test_files], conf=CONF_THRES,
                                                      imgsz=IMG_SIZE, verbose=False)
    y_pred_baseline = np.array([len(p.boxes) > 0 for p in baseline_test_preds])

    tp_b = int(np.sum(y_true & y_pred_baseline)); fn_b = int(np.sum(y_true & ~y_pred_baseline))
    fp_b = int(np.sum(~y_true & y_pred_baseline)); tn_b = int(np.sum(~y_true & ~y_pred_baseline))
    baseline_detection_accuracy = (tp_b + tn_b) / (tp_b + tn_b + fp_b + fn_b)
    baseline_detection_precision = tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else np.nan
    baseline_detection_recall = tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else np.nan
    baseline_detection_specificity = tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else np.nan
    baseline_detection_f1 = (2 * baseline_detection_precision * baseline_detection_recall /
                              (baseline_detection_precision + baseline_detection_recall)
                              if (baseline_detection_precision + baseline_detection_recall) > 0 else np.nan)
    print(f'Baseline ({BASELINE_SEG_MODEL}) image-level detection accuracy: {baseline_detection_accuracy:.4f} '
          f'(main model: {detection_accuracy:.4f})')
else:
    baseline_test_preds = None
    y_pred_baseline = None
    tp_b = fn_b = fp_b = tn_b = np.nan
    (baseline_box_map50, baseline_box_map50_95, baseline_box_precision, baseline_box_recall,
     baseline_detection_accuracy, baseline_detection_precision, baseline_detection_recall,
     baseline_detection_specificity, baseline_detection_f1) = (np.nan,) * 9
    print('TRAIN_BASELINE_COMPARISON is False -- skipping baseline detection predictions.')

### Third model comparison

Same test images, same `CONF_THRES`, the larger `yolo11l-seg` model from Section 3.

In [ ]:
if TRAIN_THIRD_MODEL:
    best_seg_third = YOLO('runs/crack_seg_third/weights/best.pt')
    seg_val_third = best_seg_third.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)
    third_box_map50    = seg_val_third.box.map50
    third_box_map50_95 = seg_val_third.box.map
    third_box_precision = seg_val_third.box.mp
    third_box_recall    = seg_val_third.box.mr

    third_test_preds = best_seg_third.predict(source=[str(p) for p in test_files], conf=CONF_THRES,
                                                imgsz=IMG_SIZE, verbose=False)
    y_pred_third = np.array([len(p.boxes) > 0 for p in third_test_preds])

    tp_t = int(np.sum(y_true & y_pred_third)); fn_t = int(np.sum(y_true & ~y_pred_third))
    fp_t = int(np.sum(~y_true & y_pred_third)); tn_t = int(np.sum(~y_true & ~y_pred_third))
    detection_accuracy_third = (tp_t + tn_t) / (tp_t + tn_t + fp_t + fn_t)
    detection_precision_third = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else np.nan
    detection_recall_third = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else np.nan
    detection_specificity_third = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else np.nan
    detection_f1_third = (2 * detection_precision_third * detection_recall_third /
                           (detection_precision_third + detection_recall_third)
                           if (detection_precision_third + detection_recall_third) > 0 else np.nan)
    print(f'Third model ({THIRD_SEG_MODEL}) image-level detection accuracy: {detection_accuracy_third:.4f} '
          f'(main model: {detection_accuracy:.4f})')
else:
    third_test_preds = None
    y_pred_third = None
    tp_t = fn_t = fp_t = tn_t = np.nan
    (third_box_map50, third_box_map50_95, third_box_precision, third_box_recall,
     detection_accuracy_third, detection_precision_third, detection_recall_third,
     detection_specificity_third, detection_f1_third) = (np.nan,) * 9
    print('TRAIN_THIRD_MODEL is False -- skipping third-model detection predictions.')

## 6. Evaluate — crack shape segmentation accuracy

Mask mAP/precision/recall come straight out of the same `val()` call (segmentation
metrics are computed alongside detection metrics for a `-seg` model). We also
compute the mean IoU between each predicted mask and its best-matching ground-truth
mask, which is a more intuitive "how good is the crack shape" number.

In [ ]:
seg_map50    = seg_val.seg.map50
seg_map50_95 = seg_val.seg.map
seg_precision = seg_val.seg.mp
seg_recall    = seg_val.seg.mr
print(f'Mask segmentation — mAP50: {seg_map50:.4f}  mAP50-95: {seg_map50_95:.4f}  '
      f'Precision: {seg_precision:.4f}  Recall: {seg_recall:.4f}')

In [ ]:
def polygon_mask(segmentation, w, h):
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    for seg in segmentation:
        pts = list(zip(seg[0::2], seg[1::2]))
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(mask, dtype=bool)

def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union > 0 else 0.0

id_by_name = {im['file_name']: im for im in coco_test['images']}
anns_by_image_test = defaultdict(list)
for a in coco_test['annotations']:
    anns_by_image_test[a['image_id']].append(a)

def best_matched_ious(preds):
    """Given a list of Results already predicted over test_files (reusing the inference
    already run in Section 5 -- no need to run the model over the test set again), return,
    for every ground-truth crack instance (in a fixed deterministic order), the
    best-matching IoU. Calling this with different models' predictions yields arrays that
    line up index-for-index on the same ground-truth instances -- exactly what Table 7's
    paired tests need."""
    ious_list = []
    for p in preds:
        fname = Path(p.path).name
        im_meta = id_by_name.get(fname)
        if im_meta is None:
            continue
        w, h = im_meta['width'], im_meta['height']
        gt_masks = [polygon_mask(a['segmentation'], w, h) for a in anns_by_image_test.get(im_meta['id'], [])]
        if not gt_masks:
            continue
        if p.masks is None:
            ious_list.extend([0.0] * len(gt_masks))  # missed every ground-truth crack in this image
            continue
        # p.masks.xy gives polygon vertices already in original-image pixel coordinates,
        # which avoids any ambiguity about the resolution of p.masks.data.
        pred_masks = [polygon_mask([poly.reshape(-1).tolist()], w, h) for poly in p.masks.xy]
        used = set()
        for gt in gt_masks:
            best_iou, best_j = 0.0, None
            for j, pm in enumerate(pred_masks):
                if j in used:
                    continue
                iou = mask_iou(gt, pm)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_j is not None:
                used.add(best_j)
            ious_list.append(best_iou)
    return np.array(ious_list)

ious = best_matched_ious(main_test_preds)
print(f'Matched {len(ious)} ground-truth crack instances across {len(test_files)} test images')
print(f'Mean mask IoU: {ious.mean():.4f}   Median: {np.median(ious):.4f}')
print(f'Instances with IoU > 0.5 (good shape match): {(ious > 0.5).mean():.4f}')

plt.figure(figsize=(6, 4))
plt.hist(ious, bins=20, edgecolor='black')
plt.xlabel('Mask IoU (predicted vs. ground truth)'); plt.ylabel('Number of crack instances')
plt.title('Segmentation shape accuracy — IoU distribution'); plt.tight_layout(); plt.show()

### Baseline model comparison

Reuses `best_matched_ious` and the baseline predictions already computed in Section 5
(no extra inference pass) over the exact same ground-truth crack instances in the
exact same order -- so `ious` and `ious_baseline` are properly paired for Tables 5 and 7. Mask
precision/recall/mAP come from the same `seg_val_baseline.val()` call from Section 5.

In [ ]:
if TRAIN_BASELINE_COMPARISON:
    ious_baseline = best_matched_ious(baseline_test_preds)
    baseline_seg_map50    = seg_val_baseline.seg.map50
    baseline_seg_map50_95 = seg_val_baseline.seg.map
    baseline_seg_precision = seg_val_baseline.seg.mp
    baseline_seg_recall    = seg_val_baseline.seg.mr
    print(f'Baseline ({BASELINE_SEG_MODEL}) mean mask IoU: {ious_baseline.mean():.4f} '
          f'(main model: {ious.mean():.4f})')
else:
    ious_baseline = None
    (baseline_seg_map50, baseline_seg_map50_95, baseline_seg_precision, baseline_seg_recall) = (np.nan,) * 4
    print('TRAIN_BASELINE_COMPARISON is False -- skipping baseline segmentation predictions.')

### Third model comparison

Same idea, reusing the third model's Section 5 predictions -- so `ious` and
`ious_third` are paired for Tables 5 and 7.

In [ ]:
if TRAIN_THIRD_MODEL:
    ious_third = best_matched_ious(third_test_preds)
    third_seg_map50    = seg_val_third.seg.map50
    third_seg_map50_95 = seg_val_third.seg.map
    third_seg_precision = seg_val_third.seg.mp
    third_seg_recall    = seg_val_third.seg.mr
    print(f'Third model ({THIRD_SEG_MODEL}) mean mask IoU: {ious_third.mean():.4f} '
          f'(main model: {ious.mean():.4f})')
else:
    ious_third = None
    (third_seg_map50, third_seg_map50_95, third_seg_precision, third_seg_recall) = (np.nan,) * 4
    print('TRAIN_THIRD_MODEL is False -- skipping third-model segmentation predictions.')

## 7. Evaluate — crack direction classification accuracy

In [ ]:
best_cls = YOLO('runs/crack_direction_cls/weights/best.pt')
cls_val = best_cls.val(data=str(DIR_DATASET), split='test')
print(f'Top-1 accuracy: {cls_val.top1:.4f}   Top-5 accuracy: {cls_val.top5:.4f}')

In [ ]:
class_names = best_cls.names  # {index: class_name}
name_to_idx = {v: k for k, v in class_names.items()}

test_dir_dataset = DIR_DATASET / 'test'
y_true_dir, y_pred_dir, files = [], [], []
for cls_name in sorted(os.listdir(test_dir_dataset)):
    for f in (test_dir_dataset / cls_name).glob('*.jpg'):
        files.append(f)
        y_true_dir.append(cls_name)

preds = best_cls.predict(source=[str(f) for f in files], imgsz=224, verbose=False)
for p in preds:
    y_pred_dir.append(class_names[int(p.probs.top1)])

print(classification_report(y_true_dir, y_pred_dir, digits=4))

labels_order = sorted(set(y_true_dir) | set(y_pred_dir))
cm = confusion_matrix(y_true_dir, y_pred_dir, labels=labels_order)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_order, yticklabels=labels_order)
plt.xlabel('Predicted direction'); plt.ylabel('True direction')
plt.title('Direction classification — confusion matrix'); plt.tight_layout(); plt.show()

### Baseline model comparison

Same `files` list and order as above, so `y_pred_dir` and `y_pred_dir_baseline` line
up image-for-image -- needed for Table 6's descriptives and Table 7's paired tests.

In [ ]:
if TRAIN_BASELINE_COMPARISON:
    best_cls_baseline = YOLO('runs/crack_direction_cls_baseline/weights/best.pt')
    class_names_baseline = best_cls_baseline.names
    preds_baseline = best_cls_baseline.predict(source=[str(f) for f in files], imgsz=224, verbose=False)
    y_pred_dir_baseline = [class_names_baseline[int(p.probs.top1)] for p in preds_baseline]
    baseline_dir_accuracy = (np.array(y_true_dir) == np.array(y_pred_dir_baseline)).mean()
    main_dir_accuracy = (np.array(y_true_dir) == np.array(y_pred_dir)).mean()
    baseline_report = classification_report(y_true_dir, y_pred_dir_baseline, output_dict=True, digits=4)
    baseline_macro = baseline_report['macro avg']
    print(f'Baseline ({BASELINE_CLS_MODEL}) top-1 accuracy: {baseline_dir_accuracy:.4f} '
          f'(main model: {main_dir_accuracy:.4f})')
else:
    y_pred_dir_baseline = None
    baseline_dir_accuracy = np.nan
    baseline_macro = {'precision': np.nan, 'recall': np.nan, 'f1-score': np.nan}
    print('TRAIN_BASELINE_COMPARISON is False -- skipping baseline direction predictions.')

### Third model comparison

Same `files` list and order, so `y_pred_dir_third` lines up image-for-image with the
other models' predictions -- needed for Table 6's descriptives and Table 7's paired tests.

In [ ]:
if TRAIN_THIRD_MODEL:
    best_cls_third = YOLO('runs/crack_direction_cls_third/weights/best.pt')
    class_names_third = best_cls_third.names
    preds_third = best_cls_third.predict(source=[str(f) for f in files], imgsz=224, verbose=False)
    y_pred_dir_third = [class_names_third[int(p.probs.top1)] for p in preds_third]
    third_dir_accuracy = (np.array(y_true_dir) == np.array(y_pred_dir_third)).mean()
    third_report = classification_report(y_true_dir, y_pred_dir_third, output_dict=True, digits=4)
    third_macro = third_report['macro avg']
    print(f'Third model ({THIRD_CLS_MODEL}) top-1 accuracy: {third_dir_accuracy:.4f} '
          f'(main model: {(np.array(y_true_dir) == np.array(y_pred_dir)).mean():.4f})')
else:
    y_pred_dir_third = None
    third_dir_accuracy = np.nan
    third_macro = {'precision': np.nan, 'recall': np.nan, 'f1-score': np.nan}
    print('TRAIN_THIRD_MODEL is False -- skipping third-model direction predictions.')

## 8. Combined summary

In [ ]:
direction_accuracy = (np.array(y_true_dir) == np.array(y_pred_dir)).mean()

summary = pd.DataFrame([
    {'Task': 'Crack vs. no-crack detection', 'Metric': 'Image-level accuracy', 'Value': detection_accuracy},
    {'Task': 'Crack vs. no-crack detection', 'Metric': 'Box mAP50', 'Value': box_map50},
    {'Task': 'Crack vs. no-crack detection', 'Metric': 'Box mAP50-95', 'Value': box_map50_95},
    {'Task': 'Crack shape segmentation', 'Metric': 'Mean mask IoU', 'Value': ious.mean()},
    {'Task': 'Crack shape segmentation', 'Metric': 'Mask mAP50', 'Value': seg_map50},
    {'Task': 'Crack shape segmentation', 'Metric': 'Mask mAP50-95', 'Value': seg_map50_95},
    {'Task': 'Crack direction classification', 'Metric': 'Top-1 accuracy', 'Value': direction_accuracy},
])
summary['Value'] = summary['Value'].map(lambda v: f'{v:.4f}')
summary

## 9. Visualize sample predictions

A handful of test images with the predicted crack mask overlaid and the predicted
direction label as the title, for a quick sanity check.

In [ ]:
sample_files = random.sample(test_files, min(6, len(test_files)))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, fpath in zip(axes.ravel(), sample_files):
    seg_pred = best_seg.predict(source=str(fpath), conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)[0]
    dir_pred = best_cls.predict(source=str(fpath), imgsz=224, verbose=False)[0]
    annotated = seg_pred.plot()[:, :, ::-1]  # BGR -> RGB
    ax.imshow(annotated)
    ax.set_title(f'{fpath.name}\npredicted direction: {class_names[int(dir_pred.probs.top1)]} '
                 f'({float(dir_pred.probs.top1conf):.2f})', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 10. Export research-paper-ready tables (descriptive & inferential statistics)

Each table is built as a `pandas` DataFrame, displayed inline, and then written to
`/content/paper_tables/` as both `.csv` (spreadsheet-ready) and `.tex` (a LaTeX
`tabular` you can `\input{}` straight into a paper), bundled into `paper_tables.zip`
for download.

- **Table 0 (descriptive, Methods)** — image counts per split (total, cracked,
  crack-free): the high-level dataset-size table a Methods/Data section needs.
- **Table 1 (descriptive, Results)** — the held-out **test** split's composition in
  detail: crack-instance counts and crack size by direction, plus a `No Crack` row.
  Matches the images the model is actually scored on below.
- **Table 2 (descriptive)** — crack detection confusion matrix (TP/FN/FP/TN), one row
  per model actually trained.
- **Table 3 (descriptive)** — crack detection *image-level* metrics (Accuracy,
  Precision, Recall, Specificity, F1), computed from Table 2.
- **Table 4 (descriptive)** — crack detection *box-level* metrics (Ultralytics' own
  box-IoU-matched precision/recall/mAP) -- a different question from Table 3, kept
  separate rather than mixed into it.
- **Table 5 (descriptive)** — crack segmentation performance (mean mask IoU, mask
  precision/recall/mAP).
- **Table 6 (descriptive)** — direction classification performance (top-1 accuracy,
  macro precision/recall/F1).
- Tables 2-6 each have one column per model (`yolo11m-...`, `yolo11n-...`,
  `yolo11l-...`), populated only for models actually trained
  (`TRAIN_BASELINE_COMPARISON` / `TRAIN_THIRD_MODEL`).
- **Table 7 (inferential)** — for each task, a **symmetric** paired significance test
  between every pair of models actually trained (no model is singled out as 'the'
  baseline): with all three trained this is 3 pairwise comparisons x 3 tasks = 9 rows;
  with just the main + one other model, 3 rows. A `Higher-performing model` column
  reports which side scored higher on the point estimate -- purely descriptive, computed
  after the test, not baked into the hypothesis (the tests themselves are two-sided:
  "are these two models different", not "is A better than B").

In [ ]:
PAPER_TABLES_DIR = Path('/content/paper_tables')
PAPER_TABLES_DIR.mkdir(exist_ok=True)
exported_tables = {}  # name -> DataFrame; exported to CSV/LaTeX in the final cell of this section

def register_table(name, df):
    exported_tables[name] = df
    return df

# Actual model names, used as column headers so results are self-identifying instead of
# generic 'Main model' / 'Baseline model' labels. Defined regardless of which optional
# models were trained, so table columns stay consistently named (populated with NaN if
# that particular model's flag was False).
MAIN_SEG_LABEL = Path(SEG_MODEL).stem
BASELINE_SEG_LABEL = Path(BASELINE_SEG_MODEL).stem
THIRD_SEG_LABEL = Path(THIRD_SEG_MODEL).stem
MAIN_CLS_LABEL = Path(CLS_MODEL).stem
BASELINE_CLS_LABEL = Path(BASELINE_CLS_MODEL).stem
THIRD_CLS_LABEL = Path(THIRD_CLS_MODEL).stem

### Table 0 — Dataset split summary (for the Methods section)

How many images are in each split, and how many are crack vs. crack-free -- the level
of detail a Methods/Data section typically needs, without the per-direction breakdown.

In [ ]:
all_coco = {split: json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json')) for split in SPLITS}

area_records = []
no_crack_counts = {split: 0 for split in SPLITS}
for split, coco in all_coco.items():
    imgs_by_id = {im['id']: im for im in coco['images']}
    for im in coco['images']:
        if im.get('direction') == 'None':
            no_crack_counts[split] += 1
    for a in coco['annotations']:
        im = imgs_by_id[a['image_id']]
        area_records.append({
            'split': split,
            'direction': im.get('direction', 'Unknown'),
            'area_px': a['area'],
            'area_pct': 100 * a['area'] / (im['width'] * im['height']),
        })
area_df = pd.DataFrame(area_records)

table0 = pd.DataFrame([
    {'split': split, 'n_images': len(all_coco[split]['images']),
     'n_cracked_images': len(all_coco[split]['images']) - no_crack_counts[split],
     'n_no_crack_images': no_crack_counts[split]}
    for split in SPLITS
])
totals = table0[['n_images', 'n_cracked_images', 'n_no_crack_images']].sum().to_dict()
table0 = pd.concat([table0, pd.DataFrame([{'split': 'All', **totals}])], ignore_index=True)
register_table('Table0_dataset_split_summary', table0)
display(table0)

### Table 1 — Test split composition (for the Results section)

The fine-grained breakdown of the held-out **test** split only -- crack-instance
counts and crack size (as % of image area) by direction, plus a `No Crack` row for
the negatives, matching the actual images the model is evaluated on in the results
below. (See Table 0 above for train/valid/test totals.)

In [ ]:
image_counts = direction_df.groupby(['split', 'direction']).size().rename('n_images').reset_index()
ann_counts = (direction_df.groupby(['split', 'direction'])['num_annotations'].sum()
              .rename('n_annotations').reset_index())
area_stats = (area_df.groupby(['split', 'direction'])['area_pct']
              .agg(['count', 'mean', 'std', 'median', 'min', 'max']).reset_index())
area_stats.columns = ['split', 'direction', 'n_instances', 'mean_area_pct', 'sd_area_pct',
                       'median_area_pct', 'min_area_pct', 'max_area_pct']

table1_cracked = (image_counts.merge(ann_counts, on=['split', 'direction'])
                  .merge(area_stats, on=['split', 'direction']))

no_crack_rows = pd.DataFrame([
    {'split': split, 'direction': 'No Crack', 'n_images': no_crack_counts[split], 'n_annotations': 0,
     'n_instances': 0, 'mean_area_pct': np.nan, 'sd_area_pct': np.nan, 'median_area_pct': np.nan,
     'min_area_pct': np.nan, 'max_area_pct': np.nan}
    for split in SPLITS
])

table1_all_splits = (pd.concat([table1_cracked, no_crack_rows], ignore_index=True)
                      .sort_values(['split', 'direction']).reset_index(drop=True).round(3))
table1 = (table1_all_splits[table1_all_splits['split'] == 'test']
          .drop(columns='split').reset_index(drop=True))
register_table('Table1_test_split_composition', table1)
display(table1)

### Table 2 — Crack detection confusion matrix

The raw TP/FN/FP/TN counts behind every image-level detection metric below, one row
per model actually trained, over the full test set (crack + crack-free images).

In [ ]:
cm_rows = [{'Model': MAIN_SEG_LABEL, 'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn, 'N': len(y_true)}]
if TRAIN_BASELINE_COMPARISON:
    cm_rows.append({'Model': BASELINE_SEG_LABEL, 'TP': tp_b, 'FN': fn_b, 'FP': fp_b, 'TN': tn_b, 'N': len(y_true)})
if TRAIN_THIRD_MODEL:
    cm_rows.append({'Model': THIRD_SEG_LABEL, 'TP': tp_t, 'FN': fn_t, 'FP': fp_t, 'TN': tn_t, 'N': len(y_true)})
table2_cm = pd.DataFrame(cm_rows)
register_table('Table2_detection_confusion_matrix', table2_cm)
display(table2_cm)

### Table 3 — Crack detection: image-level metrics

Accuracy/Precision/Recall/Specificity/F1, computed from Table 2's confusion matrix --
answers "crack present or not, per image." Columns are named for the actual models
being compared; the baseline/third columns are only populated if their respective
`TRAIN_*` flag is `True`.

In [ ]:
table3_det_img = pd.DataFrame([
    {'Metric': 'Accuracy', MAIN_SEG_LABEL: detection_accuracy, BASELINE_SEG_LABEL: baseline_detection_accuracy, THIRD_SEG_LABEL: detection_accuracy_third},
    {'Metric': 'Precision', MAIN_SEG_LABEL: detection_precision, BASELINE_SEG_LABEL: baseline_detection_precision, THIRD_SEG_LABEL: detection_precision_third},
    {'Metric': 'Recall', MAIN_SEG_LABEL: detection_recall, BASELINE_SEG_LABEL: baseline_detection_recall, THIRD_SEG_LABEL: detection_recall_third},
    {'Metric': 'Specificity', MAIN_SEG_LABEL: detection_specificity, BASELINE_SEG_LABEL: baseline_detection_specificity, THIRD_SEG_LABEL: detection_specificity_third},
    {'Metric': 'F1-score', MAIN_SEG_LABEL: detection_f1, BASELINE_SEG_LABEL: baseline_detection_f1, THIRD_SEG_LABEL: detection_f1_third},
]).round(4)
register_table('Table3_detection_image_level_metrics', table3_det_img)
display(table3_det_img)

### Table 4 — Crack detection: box-level metrics

Ultralytics' own object-detection metrics -- whether individual predicted boxes line
up with ground-truth boxes by IoU. A different question from Table 3 (box-level vs.
image-level), so read separately rather than side by side.

In [ ]:
table4_det_box = pd.DataFrame([
    {'Metric': 'Box precision', MAIN_SEG_LABEL: box_precision, BASELINE_SEG_LABEL: baseline_box_precision, THIRD_SEG_LABEL: third_box_precision},
    {'Metric': 'Box recall', MAIN_SEG_LABEL: box_recall, BASELINE_SEG_LABEL: baseline_box_recall, THIRD_SEG_LABEL: third_box_recall},
    {'Metric': 'Box mAP50', MAIN_SEG_LABEL: box_map50, BASELINE_SEG_LABEL: baseline_box_map50, THIRD_SEG_LABEL: third_box_map50},
    {'Metric': 'Box mAP50-95', MAIN_SEG_LABEL: box_map50_95, BASELINE_SEG_LABEL: baseline_box_map50_95, THIRD_SEG_LABEL: third_box_map50_95},
]).round(4)
register_table('Table4_detection_box_level_metrics', table4_det_box)
display(table4_det_box)

### Table 5 — Crack segmentation performance (descriptive)

In [ ]:
table5_seg = pd.DataFrame([
    {'Metric': 'Mean mask IoU', MAIN_SEG_LABEL: ious.mean(),
     BASELINE_SEG_LABEL: (ious_baseline.mean() if ious_baseline is not None else np.nan),
     THIRD_SEG_LABEL: (ious_third.mean() if ious_third is not None else np.nan)},
    {'Metric': 'Mask precision', MAIN_SEG_LABEL: seg_precision, BASELINE_SEG_LABEL: baseline_seg_precision, THIRD_SEG_LABEL: third_seg_precision},
    {'Metric': 'Mask recall', MAIN_SEG_LABEL: seg_recall, BASELINE_SEG_LABEL: baseline_seg_recall, THIRD_SEG_LABEL: third_seg_recall},
    {'Metric': 'Mask mAP50', MAIN_SEG_LABEL: seg_map50, BASELINE_SEG_LABEL: baseline_seg_map50, THIRD_SEG_LABEL: third_seg_map50},
    {'Metric': 'Mask mAP50-95', MAIN_SEG_LABEL: seg_map50_95, BASELINE_SEG_LABEL: baseline_seg_map50_95, THIRD_SEG_LABEL: third_seg_map50_95},
]).round(4)
register_table('Table5_segmentation_performance', table5_seg)
display(table5_seg)

### Table 6 — Direction classification performance (descriptive)

In [ ]:
report_dict = classification_report(y_true_dir, y_pred_dir, output_dict=True, digits=4)
macro = report_dict['macro avg']
y_true_dir_arr, y_pred_dir_arr = np.array(y_true_dir), np.array(y_pred_dir)
dir_acc = (y_true_dir_arr == y_pred_dir_arr).mean()

table6_dir = pd.DataFrame([
    {'Metric': 'Top-1 accuracy', MAIN_CLS_LABEL: dir_acc, BASELINE_CLS_LABEL: baseline_dir_accuracy, THIRD_CLS_LABEL: third_dir_accuracy},
    {'Metric': 'Macro precision', MAIN_CLS_LABEL: macro['precision'], BASELINE_CLS_LABEL: baseline_macro['precision'], THIRD_CLS_LABEL: third_macro['precision']},
    {'Metric': 'Macro recall', MAIN_CLS_LABEL: macro['recall'], BASELINE_CLS_LABEL: baseline_macro['recall'], THIRD_CLS_LABEL: third_macro['recall']},
    {'Metric': 'Macro F1', MAIN_CLS_LABEL: macro['f1-score'], BASELINE_CLS_LABEL: baseline_macro['f1-score'], THIRD_CLS_LABEL: third_macro['f1-score']},
]).round(4)
register_table('Table6_direction_performance', table6_dir)
display(table6_dir)

### Table 7 — Are the models significantly different from each other? (inferential)

Every model actually trained (main, and whichever of baseline/third were also
trained) was trained and scored on the *exact same data and test set*, so every
comparison here is **paired** -- both models are scored on the identical images /
crack instances, which controls for test-set variability. Unlike a one-sided
"is the main model better than the baseline" framing, this table runs a genuinely
**symmetric, two-sided test for every pair of trained models**: it asks "are these
two significantly different", not "is A better than B" -- no model is singled out as
the reference.

| Task | Test | Why this test |
|---|---|---|
| Detection | McNemar's test (two-sided) | standard test for two classifiers scored on the same items |
| Segmentation | Wilcoxon signed-rank (two-sided) | paired, non-parametric, robust to IoU being bounded/skewed |
| Direction | McNemar's test (two-sided) | same as detection -- both are per-image correct/incorrect outcomes |

**How to read it:** `p < 0.05` means the two models' paired outcomes differ
significantly -- a real difference, not just noise from a particular test split. The
`Higher-performing model` column is purely descriptive (computed from the two point
estimates *after* the test runs), not part of the hypothesis being tested. With all
three models trained, every task gets 3 pairwise rows (A-B, A-C, B-C) -- 9 rows total;
with just two models trained, 3 rows.

In [ ]:
def mcnemar_test(correct_a, correct_b):
    """Two-sided McNemar's test: are two models' per-item correctness significantly
    different on the same paired items? Uses the exact binomial form when the number of
    discordant pairs is small (n<25, the standard recommendation), and the
    continuity-corrected chi-square approximation otherwise. Symmetric in A/B -- neither
    side is treated as a reference/baseline."""
    correct_a, correct_b = np.asarray(correct_a), np.asarray(correct_b)
    b = int(np.sum(correct_a & ~correct_b))   # A right, B wrong
    c = int(np.sum(~correct_a & correct_b))   # A wrong, B right
    n = b + c
    if n == 0:
        return np.nan, 1.0, b, c
    if n < 25:
        p = stats.binomtest(b, n, 0.5, alternative='two-sided').pvalue
        stat = np.nan
    else:
        stat = (abs(b - c) - 1) ** 2 / n
        p = 1 - stats.chi2.cdf(stat, df=1)
    return stat, p, b, c

def higher_performing(label_a, score_a, label_b, score_b):
    if score_a == score_b:
        return 'Tie'
    return label_a if score_a > score_b else label_b

# Whichever models were actually trained, in training order -- Table 7 compares every
# pair among them, with no model treated as a fixed reference.
seg_models = [(MAIN_SEG_LABEL, y_true == y_pred, detection_accuracy, ious, ious.mean())]
if TRAIN_BASELINE_COMPARISON:
    seg_models.append((BASELINE_SEG_LABEL, y_true == y_pred_baseline, baseline_detection_accuracy,
                        ious_baseline, ious_baseline.mean()))
if TRAIN_THIRD_MODEL:
    seg_models.append((THIRD_SEG_LABEL, y_true == y_pred_third, detection_accuracy_third,
                        ious_third, ious_third.mean()))

cls_models = [(MAIN_CLS_LABEL, y_true_dir_arr == y_pred_dir_arr, dir_acc)]
if TRAIN_BASELINE_COMPARISON:
    cls_models.append((BASELINE_CLS_LABEL, y_true_dir_arr == np.array(y_pred_dir_baseline), baseline_dir_accuracy))
if TRAIN_THIRD_MODEL:
    cls_models.append((THIRD_CLS_LABEL, y_true_dir_arr == np.array(y_pred_dir_third), third_dir_accuracy))

if len(seg_models) < 2:
    print('Only one model was trained (TRAIN_BASELINE_COMPARISON and TRAIN_THIRD_MODEL are both False) -- '
          'nothing to pairwise-compare. Table 7 skipped.')
else:
    rows = []

    for (label_a, correct_a, acc_a, _, _), (label_b, correct_b, acc_b, _, _) in itertools.combinations(seg_models, 2):
        stat, p, b, c = mcnemar_test(correct_a, correct_b)
        rows.append({'Task': 'Crack detection', 'Model A': label_a, 'Model B': label_b,
                     'Model A score': round(acc_a, 4), 'Model B score': round(acc_b, 4),
                     'Test': "McNemar's test", 'Statistic': stat, 'p_value': p,
                     'Significant (p<0.05)': p < 0.05,
                     'Higher-performing model': higher_performing(label_a, acc_a, label_b, acc_b)})

    for (label_a, _, _, ious_a, mean_a), (label_b, _, _, ious_b, mean_b) in itertools.combinations(seg_models, 2):
        iou_diffs = ious_a - ious_b
        if np.any(iou_diffs != 0):
            seg_stat, seg_p = stats.wilcoxon(iou_diffs, alternative='two-sided')
        else:
            seg_stat, seg_p = np.nan, np.nan
        rows.append({'Task': 'Crack segmentation', 'Model A': label_a, 'Model B': label_b,
                     'Model A score': round(mean_a, 4), 'Model B score': round(mean_b, 4),
                     'Test': 'Wilcoxon signed-rank', 'Statistic': seg_stat, 'p_value': seg_p,
                     'Significant (p<0.05)': seg_p < 0.05,
                     'Higher-performing model': higher_performing(label_a, mean_a, label_b, mean_b)})

    for (label_a, correct_a, acc_a), (label_b, correct_b, acc_b) in itertools.combinations(cls_models, 2):
        stat, p, b, c = mcnemar_test(correct_a, correct_b)
        rows.append({'Task': 'Direction classification', 'Model A': label_a, 'Model B': label_b,
                     'Model A score': round(acc_a, 4), 'Model B score': round(acc_b, 4),
                     'Test': "McNemar's test", 'Statistic': stat, 'p_value': p,
                     'Significant (p<0.05)': p < 0.05,
                     'Higher-performing model': higher_performing(label_a, acc_a, label_b, acc_b)})

    table7 = pd.DataFrame(rows)
    table7[['Statistic', 'p_value']] = table7[['Statistic', 'p_value']].round(4)
    register_table('Table7_pairwise_model_comparisons', table7)
    display(table7)

### Export tables to CSV + LaTeX and download

In [ ]:
for name, df in exported_tables.items():
    df.to_csv(PAPER_TABLES_DIR / f'{name}.csv', index=False)
    try:
        (PAPER_TABLES_DIR / f'{name}.tex').write_text(
            df.to_latex(index=False, float_format='%.4f', na_rep='--'))
    except Exception as e:
        print(f'Could not export {name} to LaTeX ({e}); CSV was still written.')

print(f'Exported {len(exported_tables)} tables to {PAPER_TABLES_DIR}:')
for f in sorted(PAPER_TABLES_DIR.iterdir()):
    print(' ', f.name)

zip_path = '/content/paper_tables.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in PAPER_TABLES_DIR.iterdir():
        zf.write(f, arcname=f.name)

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except Exception as e:
    print(f'Automatic browser download not available in this environment ({e}). '
          f'The files are still saved at {PAPER_TABLES_DIR} and {zip_path}.')

## 11. (Optional) Save trained weights & tables to Google Drive

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = '/content/drive/MyDrive/crack_models'
    os.makedirs(out_dir, exist_ok=True)
    shutil.copy('runs/crack_seg/weights/best.pt', f'{out_dir}/crack_seg_best.pt')
    shutil.copy('runs/crack_direction_cls/weights/best.pt', f'{out_dir}/crack_direction_cls_best.pt')
    tables_out_dir = f'{out_dir}/paper_tables'
    if os.path.isdir(tables_out_dir):
        shutil.rmtree(tables_out_dir)
    shutil.copytree(PAPER_TABLES_DIR, tables_out_dir)
    print(f'Saved weights and paper tables to {out_dir}')
else:
    print('SAVE_TO_DRIVE is False — skipping. Weights remain at runs/crack_seg/weights/best.pt '
          'and runs/crack_direction_cls/weights/best.pt, and exported tables remain at '
          f'{PAPER_TABLES_DIR} / /content/paper_tables.zip for this session.')